# 现场 QuickTest：看一次结果，再比较前后两次录制

这里比较三个已经训练好的模型：历史多人通用模型（pooled）、当前受试者的个人模型（personal）、加入作者数据的六通道模型（mixed-common6）。它们帮助观察预测方向，不能独立证明人的真实注意力变化。

本轮数据属于 LAB_FEEDBACK（看模型反馈、再调整状态的探索数据），包括尚未看反馈的 feedback0；都不是最终测试集，也不默认进入训练。详见 [数据规则](../data/exploratory/lab_feedback/README.md) 和 [模型字典](../docs/MODEL_CATALOG.md)。

## 先运行下面的导入单元格

Notebook 只负责填写路径、调用 helper 和显示。pooled/personal 分别使用完整 24 个 EEG 通道的 **240 维**特征；mixed-common6 使用 `F7,F3,P7(T5),O1,O2,P8(T6)` 的 **60 维**特征。两路都调用同一个滤波和 Welch 频带特征实现。

只执行 load → transform → predict → compare；不执行 fit、fit_transform、微调、calibration（模型校准）、阈值调整或模型覆盖。

In [7]:
from pathlib import Path
import sys
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "scripts" / "subject_model_utils.py").is_file():
    if REPO_ROOT.parent == REPO_ROOT:
        raise FileNotFoundError("请从 EEGAttention 根目录或 notebooks 目录启动")
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "scripts"))
from subject_model_utils import run_quick_test, compare_quick_tests

print("仓库:", REPO_ROOT)
print("结果仅保存在内存；不会写回原始数据、模型或正式实验结果。")


仓库: C:\CHLight\0-Plan\EEGreproduction\EEGAttention
结果仅保存在内存；不会写回原始数据、模型或正式实验结果。


## 模式 A：单 EDF，第一段录完立即测

只替换 `EDF_PATH`。文件名：`subject_status_timestamp_extra.edf`，例如 `lyc_focus_202609141630_feedback0.edf`。同名 EDF/CSV/DSI 保持同一 stem。

`lyc/zyf` 加载对应 personal；`zqd/unknown` 运行 pooled 与 mixed-common6，并明确跳过 personal。未知标签显示 N/A。真值由文件名提取，只用于计分，不作为模型输入。

下面留空避免误跑旧测试集。检查界面可手动使用已有 `data/locked/2026-09-07/lyc_focus1_20260907.edf`，但这里只按全 EDF 计算，不采用正式清单的首尾缓冲，因此不能覆盖或对照为原正式分数。

In [10]:
EDF_PATH = r"C:\Users\13313\Documents\xwechat_files\wxid_fml5icwedmtk12_c811\msg\file\2026-09\zyf\zyf\zyf_focus1_20260914.edf"
if EDF_PATH:
    RESULT = run_quick_test(EDF_PATH)
    display(RESULT["display_table"].style.format(
        {"Accuracy": "{:.2%}", "Focus比例": "{:.2%}", "Unfocus比例": "{:.2%}"}, na_rep="N/A"
    ))
else:
    print("请填写 EDF_PATH 后执行此单元格。")


c:\CHLight\1-Workconfig\Miniconda\envs\EEG\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.9.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\CHLight\1-Workconfig\Miniconda\envs\EEG\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator PCA from version 1.9.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\CHLight\1-Workconfig\Miniconda\envs\EEG\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator SVC from version 1.9.1 when using version 1.

EDF: zyf_focus1_20260914.edf
subject=zyf; true label=focus
duration=925.50s; windows=461; pooled/personal=240维; mixed-common6=60维
                 Model                              这个模型是什么 True label Accuracy Focus比例 Unfocus比例
Existing pooled frozen                      历史多受试者通用基线（已冻结）      focus   10.63%  10.63%    89.37%
          zyf personal                        只用 zyf 历史数据训练      focus    4.56%   4.56%    95.44%
         mixed-common6 lyc+zyf 历史数据 + 作者23个 recording，共同6通道      focus    8.03%   8.03%    91.97%
现场指标按全 EDF 计算；文件名标签是 intended label（预期状态），不是独立测量出的心理真值。
mixed-common6：exploratory / channel-aligned but reference compatibility uncertain；Our reference=Pz，author reference=unknown。


,Model,这个模型是什么,True label,Accuracy,Focus比例,Unfocus比例
0,Existing pooled frozen,历史多受试者通用基线（已冻结）,focus,10.63%,10.63%,89.37%
1,zyf personal,只用 zyf 历史数据训练,focus,4.56%,4.56%,95.44%
2,mixed-common6,lyc+zyf 历史数据 + 作者23个 recording，共同6通道,focus,8.03%,8.03%,91.97%


## 模式 B：第二段录完，比较 Before / After

依次填写两次实际录制路径，例如同一个人的 `focus ... feedback0` 与 `focus ... feedback1`。当两段受试者和二分类标签一致时，显示 Accuracy 的差值（百分点）和目标状态预测比例；focus 实验的目标是 focus，unfocus 实验的目标是 unfocus。

标签不同、受试者不同/未知或重复输入同一文件时，不计算改善 Δ。单标签录制中 Accuracy 与目标预测比例数值相同，不是两项独立证据。前后比较有观察顺序、任务、时长等干扰因素，不构成因果或显著性检验。

In [ ]:
EDF_PATH_BEFORE = r"C:\Users\13313\Documents\xwechat_files\wxid_fml5icwedmtk12_c811\msg\file\2026-09\zyf\zyf\zyf_unfocus1_20260914.edf"
EDF_PATH_AFTER = r"C:\Users\13313\Documents\xwechat_files\wxid_fml5icwedmtk12_c811\msg\file\2026-09\zyf(1)\zyf\zyf_unfocus_202609141707.edf"
if EDF_PATH_BEFORE and EDF_PATH_AFTER:
    COMPARISON = compare_quick_tests(EDF_PATH_BEFORE, EDF_PATH_AFTER)
    display(COMPARISON["comparison"].style.format({
        "Before Accuracy": "{:.2%}", "After Accuracy": "{:.2%}",
        "Δ Accuracy": lambda x: f"{x * 100:+.2f} 个百分点", "Before target比例": "{:.2%}",
        "After target比例": "{:.2%}",
    }, na_rep="N/A"))
else:
    print("第二段录完后填写 EDF_PATH_BEFORE 和 EDF_PATH_AFTER。")


c:\CHLight\1-Workconfig\Miniconda\envs\EEG\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.9.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\CHLight\1-Workconfig\Miniconda\envs\EEG\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator PCA from version 1.9.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\CHLight\1-Workconfig\Miniconda\envs\EEG\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator SVC from version 1.9.1 when using version 1.

Before: zyf_focus1_20260914.edf; subject=zyf; true label=focus; duration=925.50s; windows=461
After: zyf_unfocus_202609141707.edf; subject=zyf; true label=unfocus; duration=1210.00s; windows=604
                 Model Before Accuracy After Accuracy Δ Accuracy Before target比例 After target比例 target
Existing pooled frozen          10.63%         87.42%        N/A             N/A            N/A   None
          zyf personal           4.56%         92.05%        N/A             N/A            N/A   None
         mixed-common6           8.03%         66.56%        N/A             N/A            N/A   None
现场指标按全 EDF 计算；文件名标签是 intended label（预期状态），不是独立测量出的心理真值。
mixed-common6：exploratory / channel-aligned but reference compatibility uncertain；Our reference=Pz，author reference=unknown。
两段标签不同或真值未知：不是同一状态的前后对照，不计算改善 Δ。
两段窗口数不同；按各自完整录制的比例比较，不按窗口一一配对。
前后差值是现场描述性反馈；不能据此证明因果改善或最终泛化。


,Model,Before Accuracy,After Accuracy,Δ Accuracy,Before target比例,After target比例,target
0,Existing pooled frozen,10.63%,87.42%,N/A,N/A,N/A,N/A
1,zyf personal,4.56%,92.05%,N/A,N/A,N/A,N/A
2,mixed-common6,8.03%,66.56%,N/A,N/A,N/A,N/A


## 如何看结果、如何保存数据

输出显示文件名、subject、标签、时长、窗口数和人能读懂的模型表。`RESULT['predictions']` 可查看逐窗口预测；`COMPARISON['before']` / `['after']` 保留两段的完整结果。

当前 Our reference（电压参考电极）=Pz，author reference 未知；mixed-common6 仍是 exploratory / channel-aligned but reference compatibility uncertain。其变化可能同时受受试者、录制、设备、任务、处理表示及参考电极差异影响。

今天数据放 `data/exploratory/lab_feedback/2026-09-14/`。保留原始 EDF/CSV/DSI，确认反馈轮次，计算 SHA-256 后登记 metadata。feedback0 的 `feedback_seen_before_recording=false`；feedback1+ 为 true。即便某段预测很好，也不能直接加入历史训练清单或 LOCKED_TEST；以后需要另做有记录的数据晋升决定。这里只建立规则，不伪造录制或保存新的实验结论。